[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jalonso1979/puremacro/blob/main/curso/notebooks/T00_B_local_model_setup.ipynb)

> **Google Colab note**: the next cell installs `puremacro` and, if needed, fetches only the `curso/notebooks` folder (helpers, frozen data and models) from the public repository [jalonso1979/puremacro](https://github.com/jalonso1979/puremacro). It does not ask for Google Drive access.

In [1]:
# Environment setup: Google Colab, local install or Juno (iPad)
import os
import sys
from pathlib import Path

def _safe_probe(p, is_dir=None):
    """Does the path exist? Never raises OSError/PermissionError in the Juno sandbox."""
    try:
        p = Path(p)
        if is_dir is True:
            return p.is_dir()
        if is_dir is False:
            return p.is_file()
        return p.exists()
    except (OSError, PermissionError, TypeError, ValueError):
        return False

if "google.colab" in sys.modules:
    !pip install -q "puremacro>=4.8.1,<5"
    if not _safe_probe(Path.cwd() / "_nbstyle.py", is_dir=False):
        # Partial clone of the public repository: only curso/notebooks, no history
        _repo = Path("/content/puremacro")
        if not _safe_probe(_repo, is_dir=True):
            !git clone -q --depth 1 --filter=blob:none --sparse https://github.com/jalonso1979/puremacro.git {_repo}
        !git -C {_repo} sparse-checkout set curso/notebooks 2>&1 | tail -n 2
        os.chdir(_repo / "curso/notebooks")

# Put the course helpers folder (_nbstyle, _datos) on sys.path, without leaving the sandbox
_candidates = [Path.cwd(), Path.cwd() / "notebooks", Path.cwd() / "curso/notebooks"]
try:
    _candidates.insert(0, Path(__file__).resolve().parent)  # Juno defines __file__
except (NameError, OSError):
    pass
for _c in _candidates:
    if _safe_probe(_c / "_nbstyle.py", is_dir=False):
        if str(_c) not in sys.path:
            sys.path.insert(0, str(_c))
        break

# Inline figures and the course style
%matplotlib inline
try:
    import _nbstyle
    _nbstyle.apply_style()
except ImportError:
    pass

# Set up your local model

This lesson tells you how to install the course's language model, Qwen3-4B, on your own device, where the labs use it to read and score text, and checks that it answers. Nothing here needs an API key or an account: the model runs on your device and, after a one-time download, without a network. If you cannot install it, cache mode is enough for every lab: each one runs to the end with the reference answers the course ships.

In [2]:
import _llm, sys, platform

# Installation steps per platform; the cell prints only the ones for your device.
# The last step of each is the model download, with its size.
PASOS = {
    "ios": "\n".join([
        "iPad (Juno beta)",
        "  1. Install the Juno beta from the TestFlight invitation your instructor sends you.",
        "  2. Install puremacro from Juno's package manager.",
        "  3. Nothing else: mlx-lm comes with the beta and downloads the model the first time you",
        "     use it, in the cells below. Keep Juno in front while it downloads.",
        "Qwen3-4B, 4-bit: about 2.5 GB",
    ]),
    "darwin": "\n".join([
        "Mac",
        "  Apple silicon (M1 chip or later):",
        "  1. Install mlx-lm in your notebooks' Python: pip install mlx-lm",
        "  2. The model downloads the first time you use it, in the cells below.",
        "  Intel processor (or if pip cannot find mlx-lm):",
        "  1. Install the Ollama app (ollama.com) and open it.",
        "  2. In a terminal, download the model: ollama pull qwen3:4b",
        "Qwen3-4B, 4-bit: about 2.5 GB",
    ]),
    "win32": "\n".join([
        "Windows",
        "  1. Install Ollama (ollama.com) and open it.",
        "     If you cannot install software, a llamafile run as a local server also works.",
        "  2. In a terminal (PowerShell), download the model: ollama pull qwen3:4b",
        "Qwen3-4B, 4-bit: about 2.5 GB",
    ]),
    "linux": "\n".join([
        "Linux",
        "  1. Install Ollama (ollama.com) and keep its server running.",
        "     If you cannot install software, a llamafile run as a local server also works.",
        "  2. In a terminal, download the model: ollama pull qwen3:4b",
        "Qwen3-4B, 4-bit: about 2.5 GB",
    ]),
    "colab": "\n".join([
        "Google Colab",
        "  Cache mode for now: every lab runs to the end from the course cache,",
        "  with nothing to install. Live models on Colab will be enabled later.",
        "Qwen3-4B, 4-bit: about 2.5 GB",
    ]),
}

if "google.colab" in sys.modules:
    plataforma = "colab"
# A Python before 3.13 on the iPad says "darwin"; platform.machine() is asked only there.
elif sys.platform == "darwin" and platform.machine().startswith(("iPad", "iPhone")):
    plataforma = "ios"
else:
    plataforma = next((p for p in PASOS if sys.platform.startswith(p)), sys.platform)
print(PASOS.get(plataforma, f"No steps for '{plataforma}': use cache mode (at the end of this lesson), "
                            "which is enough for every lab."))

Mac
  Apple silicon (M1 chip or later):
  1. Install mlx-lm in your notebooks' Python: pip install mlx-lm
  2. The model downloads the first time you use it, in the cells below.
  Intel processor (or if pip cannot find mlx-lm):
  1. Install the Ollama app (ollama.com) and open it.
  2. In a terminal, download the model: ollama pull qwen3:4b
Qwen3-4B, 4-bit: about 2.5 GB


## The download is homework

Do the installation and the download at home, before the lab session: the model is about 2.5 GB, and if the whole class downloads it at once in the classroom the network will not cope. With Ollama, the download is the `ollama pull` of the steps above; otherwise the first cell that runs the model downloads it. For the reference model, `qwen3-4b`, that is the live-comparison cell near the end of this lesson (the one that prints `reference:` and `live:`): it runs your model even when the course cache already has the answer, which the self-check cell only reads. For the small model, `qwen3-1.7b`, it is the self-check cell, because the course cache has no answers for that model. On the iPad, keep Juno in front with the screen on while both cells run: if you switch apps or the screen locks, iPadOS suspends Juno and the download stops. Use a stable Wi-Fi network without a data cap, not your phone's data; if the download breaks off, rerun the cell that was downloading (or the `ollama pull`). When the installation is done, run the next cell: it prints a short report of what the course helper finds and asks the model one question.

In [3]:
# Self-check: the reference answer, what the course helper finds, and one question to your
# model. The reference is read first, from the course cache under the reference model
# (qwen3-4b), before your model runs and can store an answer of its own.
Q = "Answer with one word: what does GDP measure?"
ref = _llm.get_llm(mode="cache", model=_llm.REFERENCE_MODEL, lang="en").try_complete(Q, max_tokens=16)
llm = _llm.get_llm(lang="en")
print(llm.report())
r = llm.try_complete(Q, max_tokens=16)
print(r if r is not None else "(no model: cache mode)")

Course local model
  mode:     cache (only the course cache; no engines, no network)
  model:    qwen3-4b
  cache:    notebooks/data_curso/llm_cache.sqlite (22 rows)
  backend:  fallback (puremacro 4.8.1; without puremacro.llm)
Economy.


## Reference answer and live answer

The self-check cell read the reference answer before your model ran: the answer of Qwen3-4B at 4 bits, computed once by the instructor and shipped in the course cache. The next cell asks your local model the same question, bypassing the cache, and prints both. The two can differ even though the model and the question are the same. Quantisation stores each weight in 4 bits and each format (MLX, GGUF) rounds in its own way; the engine (MLX, llama.cpp or Ollama) adds in a different order with different compute kernels. Those differences move the last decimals of the model's probabilities, and when two words nearly tie that is enough to change the answer. So in graded work, report both: the reference answer, which anyone can reproduce, and your live one, with the engine and the model you used. The instructor's reference exists only for the `qwen3-4b` prompts that the course cache ships: if the reference line says `None`, the course cache lacks this one, so ask the instructor. Until the course cache has it, the reference line of a later run is not the instructor's either: if a `qwen3-4b` run of the self-check cell got an answer in auto mode, it stored that answer in the course cache file, and later reference lines print it. A `None` on the live line means there was no live answer; the `[local model]` note printed before it says why.

In [4]:
# Your local model on the same question Q, next to the reference `ref` that the self-check
# cell read. In auto mode, "off" runs the model even when the cache has the answer; in cache
# mode there is no live answer and the "live" line comes from the cache too.
modo_vivo = "off" if llm.mode == "auto" else llm.mode
vivo = _llm.get_llm(mode=modo_vivo, lang="en").try_complete(Q, max_tokens=16)
print("reference:", ref)
print("live:", vivo)

reference: Economy.
live: Economy.


## If nothing works

That is fine: every lab runs to the end from the course cache (`data_curso/llm_cache.sqlite`, which travels with the notebooks), with or without a model. The cache holds the instructor's `qwen3-4b` answers to the prompts the course ships; if one is missing, the lab prints `None` and a note instead of stopping, so tell the instructor. To skip the engine search altogether, run `import os; os.environ["MAV_LLM_MODE"] = "cache"` in a cell before using the model. The small model is only a fallback for devices with less than 5 GB of memory for the model (in auto mode the self-check report then warns that `qwen3-4b` does not fit): there, set `os.environ["PUREMACRO_LLM_MODEL"] = "qwen3-1.7b"`, knowing that the course cache carries no answers for it and that its answers stray further from the reference, or stay in cache mode. To ask for help, copy the self-check report (it carries no absolute paths) and ask on Canvas, the course's official channel, or at office hours.